In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
SEED = 69
import tensorflow as tf
import keras
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
print('Seed set')

I0000 00:00:1787500834.897467   14800 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787500836.756022   14800 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Seed set


In [2]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import applications, callbacks, layers, models, optimizers, regularizers, utils
warnings.filterwarnings('ignore')
print('Libraries ready')

Libraries ready


In [ ]:
import kagglehub
download_path = kagglehub.dataset_download('msambare/fer2013')

def find_split_root(base_path):
    base = Path(base_path)
    for root, dirs, files in os.walk(base):
        if 'train' in dirs and 'test' in dirs:
            return Path(root)
    return base

dataset_dir = find_split_root(download_path)
train_dir   = str(dataset_dir / 'train')
test_dir    = str(dataset_dir / 'test')
print('Train:', train_dir)
print('Test: ', test_dir)

Train: /home/ubuntu/.cache/kagglehub/datasets/msambare/fer2013/versions/1/train
Test:  /home/ubuntu/.cache/kagglehub/datasets/msambare/fer2013/versions/1/test


: 

In [ ]:
import sys
from pathlib import Path

import numpy as np
from sklearn.metrics import classification_report

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.predict import CLASS_NAMES, predict

MODEL_PATH = REPO_ROOT / 'onnx_files' / 'best_effv2s.onnx'
if not MODEL_PATH.exists():
    MODEL_PATH = Path('/opt/dlami/nvme/Emotion-Recognition/onnx_files/best_effv2s.onnx')

assert Path(train_dir).is_dir(), f'Train directory not found: {train_dir}'
assert Path(test_dir).is_dir(), f'Test directory not found: {test_dir}'
assert MODEL_PATH.is_file(), f'ONNX model not found: {MODEL_PATH}'


def evaluate_split(split_dir, model_path):
    image_paths = sorted(
        path for path in Path(split_dir).rglob('*')
        if path.is_file() and path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}
    )
    y_true = []
    y_pred = []

    for index, image_path in enumerate(image_paths, start=1):
        true_label = image_path.parent.name.lower()
        if true_label not in CLASS_NAMES:
            raise ValueError(f'Unexpected class directory: {image_path.parent.name}')
        y_true.append(true_label)
        y_pred.append(predict(image_path, str(model_path)))

        if index % 1000 == 0 or index == len(image_paths):
            print(f'{split_dir}: {index}/{len(image_paths)} images evaluated')

    print(f'\n{split_dir} accuracy: {np.mean(np.array(y_true) == np.array(y_pred)):.4f}')
    print(classification_report(y_true, y_pred, labels=CLASS_NAMES, target_names=CLASS_NAMES, zero_division=0))
    return y_true, y_pred


print(f'Using model: {MODEL_PATH}')
train_true, train_pred = evaluate_split(train_dir, MODEL_PATH)
test_true, test_pred = evaluate_split(test_dir, MODEL_PATH)